# 🏓 Fine-tune Qwen2.5-VL for Serve Detection

This notebook fine-tunes Qwen2.5-VL using **QLoRA** for efficient training on Colab Pro.

## Requirements
- **GPU**: L4 (24GB) or A100 (40GB) - Colab Pro
- **Dataset**: 200+ labeled frames (serve/not_serve)
- **Training time**: 2-4 hours

## Workflow
1. Upload your dataset (or create it here)
2. Configure training parameters
3. Train the model
4. Evaluate and save

In [ ]:
# @title 1. Install Dependencies
!pip install -q transformers>=4.45.0
!pip install -q peft>=0.13.0
!pip install -q trl>=0.12.0
!pip install -q bitsandbytes>=0.44.0
!pip install -q accelerate>=0.34.0
!pip install -q datasets>=3.0.0
!pip install -q qwen-vl-utils
!pip install -q pillow opencv-python
!pip install -q wandb  # Optional: experiment tracking

print("✅ Dependencies installed!")

In [ ]:
# @title 2. Check GPU & Login to Hugging Face
import torch
import os
from huggingface_hub import login

# ========== HUGGING FACE TOKEN ==========
# Option 1: Set your token here directly
HF_TOKEN = ""  # @param {type:"string"}

# Option 2: Or use Colab secrets (recommended)
# Go to: Colab sidebar (key icon) → Add secret → Name: HF_TOKEN, Value: your_token
# =========================================

# Try to login
if HF_TOKEN:
    login(token=HF_TOKEN)
    print("✅ Logged in with provided token")
else:
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
        login(token=token)
        print("✅ Logged in with Colab secret")
    except:
        print("⚠️ No HF token found!")
        print("   To fix this:")
        print("   1. Go to https://huggingface.co/settings/tokens")
        print("   2. Create a token (read access is enough)")
        print("   3. Either paste it in HF_TOKEN above, or:")
        print("   4. Add it to Colab secrets: sidebar key icon → Add secret → HF_TOKEN")
        print("\n   Then re-run this cell.")

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\n✅ GPU: {gpu_name} ({gpu_memory:.1f} GB)")
    
    if gpu_memory < 20:
        print("⚠️ Low GPU memory. Using 2B model.")
    elif gpu_memory >= 40:
        print("🚀 A100 detected - can train 7B model!")
    else:
        print("✅ L4/V100 detected - good for 7B with QLoRA")
else:
    print("\n❌ No GPU! Go to Runtime → Change runtime type → GPU")

---
## Part A: Dataset Preparation

You need a dataset with:
- `serve/` folder: frames showing serve situations
- `not_serve/` folder: frames showing rallies, between points, etc.

In [ ]:
# @title A1. Upload Dataset (Option 1: ZIP file)
from google.colab import files
import zipfile

print("Upload a ZIP file with this structure:")
print("  dataset.zip")
print("  ├── serve/")
print("  │   ├── frame1.jpg")
print("  │   └── ...")
print("  └── not_serve/")
print("      ├── frame1.jpg")
print("      └── ...")
print()

uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print(f"✅ Extracted to dataset/")
        
        # Show contents
        import glob
        serve_count = len(glob.glob('dataset/serve/*.jpg')) + len(glob.glob('dataset/serve/*.png'))
        not_serve_count = len(glob.glob('dataset/not_serve/*.jpg')) + len(glob.glob('dataset/not_serve/*.png'))
        print(f"   Serve frames: {serve_count}")
        print(f"   Not-serve frames: {not_serve_count}")

In [ ]:
# @title A2. Create Dataset from Video (Google Drive)
import cv2
import os
import shutil
import glob
import zipfile
from google.colab import drive

# ========== CONFIGURE ==========
DRIVE_PATH = "/content/drive/MyDrive/pickleball/video.zip"  # @param {type:"string"}
SERVE_TIMESTAMPS = "42, 59, 74, 90, 101"  # @param {type:"string"}
SERVE_WINDOW_BEFORE = 2.0  # @param {type:"number"} # Seconds before timestamp to include
FRAMES_PER_SERVE = 3  # @param {type:"integer"} # Number of frames to extract per serve window
FRAMES_PER_RALLY = 3  # @param {type:"integer"} # 3-fold: extract 3 not-serve frames per gap (to balance with serve frames)
MIN_GAP_FOR_RALLY = 5.0  # @param {type:"number"} # Minimum gap between serves to extract rally frame
# ================================

# Mount Google Drive
if not os.path.exists("/content/drive"):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("📂 Google Drive already mounted")

# Check file exists
if not os.path.exists(DRIVE_PATH):
    print(f"\n❌ File not found at: {DRIVE_PATH}")
    print("\n📁 Looking for files in your Drive...")
    for ext in ['zip', 'mp4', 'MP4', 'mkv']:
        found = glob.glob(f"/content/drive/MyDrive/**/*.{ext}", recursive=True)[:5]
        for f in found:
            print(f"   Found: {f}")
    raise Exception("Update DRIVE_PATH with the correct path")

# Handle zip or video file
video_file = None

if DRIVE_PATH.endswith('.zip'):
    # Clean previous video cache so we use the new zip, not old extracted video
    if os.path.exists('video_extracted'):
        shutil.rmtree('video_extracted')
        print("🧹 Cleared previous video cache")
    print(f"📦 Extracting zip: {DRIVE_PATH}")
    with zipfile.ZipFile(DRIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall('video_extracted')
    # Find video in extracted contents
    for ext in ['mp4', 'MP4', 'mkv', 'avi', 'mov']:
        found = glob.glob(f"video_extracted/**/*.{ext}", recursive=True)
        if not found:
            found = glob.glob(f"video_extracted/*.{ext}")
        if found:
            video_file = found[0]
            break
    if not video_file:
        print("Files extracted:")
        !ls -la video_extracted/
        raise Exception("❌ No video file found in zip!")
else:
    video_file = DRIVE_PATH

print(f"✅ Using video: {video_file}")

# === STEP 2: Process the video ===
print(f"\n📹 Processing: {video_file}")

serve_times = sorted([float(t.strip()) for t in SERVE_TIMESTAMPS.split(",") if t.strip()])

# Append to dataset (don't delete - keeps frames from previous runs)
os.makedirs("dataset/serve", exist_ok=True)
os.makedirs("dataset/not_serve", exist_ok=True)

# Prefix for this video to avoid overwriting when appending from multiple videos
video_basename = os.path.splitext(os.path.basename(video_file))[0]
prefix = "".join(c if c.isalnum() or c in "_-" else "_" for c in video_basename)[:30]

cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS)

if fps == 0:
    cap.release()
    raise Exception("❌ Cannot read video file! File may be corrupted.")

duration = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) / fps

print(f"Video: {duration:.0f}s, {fps:.0f} FPS")
print(f"Serve timestamps: {len(serve_times)}")
print(f"Window: {SERVE_WINDOW_BEFORE}s before each timestamp, {FRAMES_PER_SERVE} frames per window\n")

# === SERVE FRAMES: Multiple frames from window before timestamp ===
print("📍 Extracting SERVE frames...")
serve_count = 0
for idx, timestamp in enumerate(serve_times):
    start_t = max(0, timestamp - SERVE_WINDOW_BEFORE)
    end_t = timestamp
    
    if FRAMES_PER_SERVE == 1:
        frame_times = [end_t]
    else:
        step = (end_t - start_t) / (FRAMES_PER_SERVE - 1)
        frame_times = [start_t + i * step for i in range(FRAMES_PER_SERVE)]
    
    print(f"   Serve #{idx+1} at {timestamp}s (window: {start_t:.1f}s - {end_t:.1f}s)")
    
    for i, t in enumerate(frame_times):
        frame_idx = int(t * fps)
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            serve_count += 1
            path = f"dataset/serve/{prefix}_serve_{idx+1:03d}_f{i+1:02d}_t{t:.1f}s.jpg"
            cv2.imwrite(path, frame)
            print(f"      ✅ {os.path.basename(path)}")

# === NOT-SERVE FRAMES: 3-fold - multiple frames per gap between serves ===
not_serve_count = 0

for i in range(len(serve_times) - 1):
    t1 = serve_times[i]
    t2 = serve_times[i + 1]
    gap = t2 - t1
    
    if gap >= MIN_GAP_FOR_RALLY:
        # Extract FRAMES_PER_RALLY evenly spaced frames in the rally (between t1 and t2)
        if FRAMES_PER_RALLY == 1:
            frame_times = [(t1 + t2) / 2]
        else:
            step = gap / (FRAMES_PER_RALLY + 1)
            frame_times = [t1 + (j + 1) * step for j in range(FRAMES_PER_RALLY)]
        
        for j, t in enumerate(frame_times):
            frame_idx = int(t * fps)
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            ret, frame = cap.read()
            if ret:
                not_serve_count += 1
                cv2.imwrite(f"dataset/not_serve/{prefix}_rally_{not_serve_count:03d}_t{t:.1f}s.jpg", frame)
                print(f"   ❌ {prefix}_rally_{not_serve_count:03d} at {t:.1f}s (gap {i+1}, frame {j+1}/{FRAMES_PER_RALLY})")

cap.release()

# Count total in dataset (including previous runs)
total_serve = len(glob.glob("dataset/serve/*.jpg")) + len(glob.glob("dataset/serve/*.png"))
total_not = len(glob.glob("dataset/not_serve/*.jpg")) + len(glob.glob("dataset/not_serve/*.png"))

print(f"\n{'='*60}")
print(f"✅ Extracted this run: {serve_count} serve, {not_serve_count} not-serve")
print(f"   Total in dataset:  {total_serve} serve, {total_not} not-serve")
print(f"{'='*60}")

In [ ]:
# @title A3. Generate Training JSON (IMPROVED - Consistent prompt)
import json
import random
import glob
from pathlib import Path

DATASET_DIR = "dataset"  # @param {type:"string"}
BALANCE_RATIO = 1.0  # @param {type:"number"} # Keep balanced (1:1 ratio)

# The EXACT prompt used for training AND testing - consistency is key!
TRAINING_PROMPT = "Is this a pickleball serve situation? Answer YES or NO."

# Collect all files
serve_files = glob.glob(f"{DATASET_DIR}/serve/*.jpg") + glob.glob(f"{DATASET_DIR}/serve/*.png")
not_serve_files = glob.glob(f"{DATASET_DIR}/not_serve/*.jpg") + glob.glob(f"{DATASET_DIR}/not_serve/*.png")

print(f"📁 Found files:")
print(f"   Serve: {len(serve_files)}")
print(f"   Not-serve: {len(not_serve_files)}")

# Balance dataset (1:1 ratio for binary classification)
min_count = min(len(serve_files), len(not_serve_files))
if len(serve_files) > min_count:
    random.shuffle(serve_files)
    serve_files = serve_files[:min_count]
    print(f"\n⚖️ Balanced: reduced serve to {min_count}")
if len(not_serve_files) > min_count:
    random.shuffle(not_serve_files)
    not_serve_files = not_serve_files[:min_count]
    print(f"\n⚖️ Balanced: reduced not_serve to {min_count}")

data = []

# Serve frames - simple YES answer
for img_path in serve_files:
    data.append({
        "image": os.path.abspath(img_path),
        "label": "serve",
        "messages": [
            {"role": "user", "content": TRAINING_PROMPT},
            {"role": "assistant", "content": "YES"}
        ]
    })

# Not-serve frames - simple NO answer
for img_path in not_serve_files:
    data.append({
        "image": os.path.abspath(img_path),
        "label": "not_serve",
        "messages": [
            {"role": "user", "content": TRAINING_PROMPT},
            {"role": "assistant", "content": "NO"}
        ]
    })

# Shuffle and split (80/10/10)
random.shuffle(data)
n = len(data)
train_end = int(n * 0.8)
val_end = int(n * 0.9)

train_data = data[:train_end]
val_data = data[train_end:val_end]
test_data = data[val_end:]

# Save
with open("train_data.json", "w") as f:
    json.dump(train_data, f, indent=2)
with open("val_data.json", "w") as f:
    json.dump(val_data, f, indent=2)
with open("test_data.json", "w") as f:
    json.dump(test_data, f, indent=2)

# Also save the prompt for reference
with open("training_prompt.txt", "w") as f:
    f.write(TRAINING_PROMPT)

serve_count = sum(1 for d in data if d["label"] == "serve")
not_serve_count = len(data) - serve_count

print(f"\n📊 Final Dataset Summary:")
print(f"   Total: {len(data)} samples")
print(f"   Serve: {serve_count}")
print(f"   Not-serve: {not_serve_count}")
print(f"   Ratio: 1:1 (balanced)")
print(f"\n   Train: {len(train_data)}")
print(f"   Val: {len(val_data)}")
print(f"   Test: {len(test_data)}")
print(f"\n   Training prompt: \"{TRAINING_PROMPT}\"")

---
## Part B: Model Training

In [ ]:
# @title B1. Load Base Model with QLoRA
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"  # @param ["Qwen/Qwen2.5-VL-3B-Instruct", "Qwen/Qwen2.5-VL-7B-Instruct"]

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_NAME}...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Prepare for training
model = prepare_model_for_kbit_training(model)

# LoRA config - V3: Smaller rank to prevent overfitting on small datasets
lora_config = LoraConfig(
    r=16,              # Reduced from 64 - less capacity = less overfitting
    lora_alpha=32,     # 2x rank
    target_modules=["q_proj", "v_proj"],  # Only attention - fewer params to tune
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)
# Note: With small datasets (<100 samples), large LoRA rank causes overfitting

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"\n✅ Model loaded with LoRA!")
print(f"   GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# @title B2. Prepare Dataset for Training
from datasets import Dataset as HFDataset
from PIL import Image
import json

def load_dataset_from_json(json_file):
    """Load JSON and convert to simple format for training."""
    with open(json_file) as f:
        data = json.load(f)
    
    # Simple format: image path, question, answer
    processed = {
        "image": [],
        "question": [],
        "answer": [],
        "label": []
    }
    
    for item in data:
        processed["image"].append(item["image"])
        processed["question"].append(item["messages"][0]["content"])
        processed["answer"].append(item["messages"][1]["content"])
        processed["label"].append(item["label"])
    
    return HFDataset.from_dict(processed)

train_dataset = load_dataset_from_json("train_data.json")
val_dataset = load_dataset_from_json("val_data.json")

print(f"✅ Datasets loaded:")
print(f"   Train: {len(train_dataset)} samples")
print(f"   Val: {len(val_dataset)} samples")
print(f"   Columns: {train_dataset.column_names}")
print(f"\n   Sample:")
print(f"   Image: {train_dataset[0]['image'][:50]}...")
print(f"   Answer: {train_dataset[0]['answer'][:50]}...")

In [ ]:
# @title B3. Training Configuration (V5 - WITH EARLY STOPPING)
from transformers import TrainingArguments, EarlyStoppingCallback

# ========== CONFIGURE (V5 - Early stopping to prevent overfitting) ==========
OUTPUT_DIR = "serve-detector-lora-v5"  # @param {type:"string"}
NUM_EPOCHS = 6  # @param {type:"integer"} # Reduced - early stopping will handle it
BATCH_SIZE = 1  # @param {type:"integer"}
GRADIENT_ACCUMULATION = 4  # @param {type:"integer"}
LEARNING_RATE = 5e-4  # @param {type:"number"} # Higher LR for answer-only training
EARLY_STOPPING_PATIENCE = 4  # @param {type:"integer"} # Stop after 4 evals without improvement
# ============================================================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=25,  # Save more often to capture best model
    eval_steps=25,  # Eval more often
    eval_strategy="steps",
    save_total_limit=3,
    load_best_model_at_end=True,  # Load best checkpoint when training ends
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
)

# Early stopping callback - will stop if val loss doesn't improve
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_threshold=0.01,
)

print(f"✅ Training config (V5 - Early Stopping):")
print(f"   Epochs: {NUM_EPOCHS} (max)")
print(f"   Effective batch size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Early stopping: patience={EARLY_STOPPING_PATIENCE} evals")
print(f"   Output: {OUTPUT_DIR}")
print(f"\n   Will auto-stop when validation loss stops improving!")

In [ ]:
# @title B4. Train the Model (FIXED - Only loss on answer tokens)
from trl import SFTTrainer, SFTConfig
from qwen_vl_utils import process_vision_info
from PIL import Image

IGNORE_INDEX = -100

def collate_fn(examples):
    """FIXED: Only compute loss on answer tokens, mask all prompt/image tokens."""
    texts = []
    images_list = []
    prompt_texts = []
    
    for example in examples:
        image_path = example["image"]
        question = example["question"]
        answer = example["answer"]
        
        # Build PROMPT (without answer) - we need this to know where answer starts
        prompt_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": question}
                ]
            }
        ]
        prompt_text = processor.apply_chat_template(
            prompt_messages, 
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Build FULL text (prompt + answer)
        full_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": question}
                ]
            },
            {
                "role": "assistant",
                "content": answer
            }
        ]
        full_text = processor.apply_chat_template(
            full_messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        texts.append(full_text)
        prompt_texts.append(prompt_text)
        
        # Process image
        image_inputs, _ = process_vision_info(full_messages)
        images_list.append(image_inputs[0] if image_inputs else None)
    
    # Tokenize full batch
    batch = processor(
        text=texts,
        images=images_list,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )
    
    # Create labels - ONLY compute loss on answer tokens
    labels = batch["input_ids"].clone()
    
    for i, prompt_text in enumerate(prompt_texts):
        # Tokenize prompt WITH image to get correct token count
        prompt_batch = processor(
            text=[prompt_text],
            images=[images_list[i]],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )
        prompt_len = prompt_batch["input_ids"].shape[1]
        
        # Mask all prompt/image tokens
        labels[i, :prompt_len] = IGNORE_INDEX
    
    # Mask padding
    labels[labels == processor.tokenizer.pad_token_id] = IGNORE_INDEX
    
    batch["labels"] = labels
    return batch

# SFT Config - V5 (higher LR for answer-only training)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=5e-4,  # Higher LR since we only train on 1-2 tokens per sample
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_steps=25,
    eval_steps=25,
    eval_strategy="steps",
    save_total_limit=3,
    load_best_model_at_end=True,  # Required for EarlyStoppingCallback
    metric_for_best_model="eval_loss",  # Required for EarlyStoppingCallback
    greater_is_better=False,
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field=None,
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[early_stopping],  # Add early stopping
)

print("🚀 Starting training (V5 - with early stopping)...")
print(f"   Training samples: {len(train_dataset)}")
print(f"   Validation samples: {len(val_dataset)}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Early stopping: patience={EARLY_STOPPING_PATIENCE}")
print()

trainer.train()

print("\n✅ Training complete!")
print(f"   Best model loaded from: {trainer.state.best_model_checkpoint}")

In [ ]:
# @title B4-DIAGNOSE. Check what's wrong with training
from qwen_vl_utils import process_vision_info
import torch

print("🔍 DIAGNOSTIC: Checking training setup...\n")

# 1. Check trainable parameters
print("1️⃣ TRAINABLE PARAMETERS:")
trainable_params = []
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_params.append(name)
        print(f"   ✅ {name}: {param.shape}")
print(f"\n   Total trainable: {len(trainable_params)}")

if len(trainable_params) == 0:
    print("   ❌ NO TRAINABLE PARAMETERS! This is the problem.")

# 2. Check a single forward pass
print("\n2️⃣ FORWARD PASS TEST:")
sample = train_dataset[0]
messages = [
    {
        "role": "user", 
        "content": [
            {"type": "image", "image": sample["image"]},
            {"type": "text", "text": sample["question"]}
        ]
    },
    {"role": "assistant", "content": sample["answer"]}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
image_inputs, _ = process_vision_info(messages)

inputs = processor(
    text=[text],
    images=[image_inputs[0]],
    return_tensors="pt",
    padding=True,
)

# Move to GPU
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Create labels
labels = inputs["input_ids"].clone()
labels[labels == processor.tokenizer.pad_token_id] = -100
inputs["labels"] = labels

print(f"   Input shape: {inputs['input_ids'].shape}")
print(f"   Labels shape: {labels.shape}")
print(f"   Non-masked labels: {(labels != -100).sum().item()}")

# Forward pass
model.train()
with torch.enable_grad():
    outputs = model(**inputs)
    loss = outputs.loss
    
print(f"   Loss: {loss.item():.4f}")

# 3. Check if loss can backprop
print("\n3️⃣ GRADIENT TEST:")
loss.backward()

grad_found = False
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_norm = param.grad.norm().item()
        if grad_norm > 0:
            print(f"   ✅ {name}: grad_norm={grad_norm:.6f}")
            grad_found = True
            
if not grad_found:
    print("   ❌ NO GRADIENTS FLOWING! LoRA layers not receiving gradients.")

# 4. Check if the loss even depends on answer tokens
print("\n4️⃣ ANSWER TOKEN CHECK:")
# Get answer tokens
answer_text = sample["answer"]
answer_tokens = processor.tokenizer(answer_text, add_special_tokens=False)["input_ids"]
print(f"   Answer: '{answer_text}'")
print(f"   Answer tokens: {answer_tokens}")

# Find where answer appears in input_ids
input_ids_list = inputs["input_ids"][0].tolist()
print(f"   Total tokens: {len(input_ids_list)}")

# Check label distribution
print(f"   Labels != -100 (loss computed on): {(labels[0] != -100).sum().item()} tokens")

model.zero_grad()
print("\n" + "="*60)
print("If gradients are ~0 or non-existent, the issue is LoRA config.")
print("If gradients exist but loss doesn't drop, learning rate may be too low.")
print("="*60)

In [ ]:
# @title B4-DEBUG. Overfit Test (FIXED - Only loss on answer tokens)
# This tests if training works at all by trying to overfit on 20 samples

from trl import SFTTrainer, SFTConfig
from qwen_vl_utils import process_vision_info
import random

print("🔬 DEBUG MODE: Testing if model can learn at all...")
print("   Using 20 samples, high LR, no validation")
print("   FIX: Only computing loss on answer tokens (not 2700+ image tokens!)\n")

# Take tiny subset - 10 serve, 10 not-serve
tiny_serve = [ex for ex in train_dataset if ex["answer"] == "YES"][:10]
tiny_not = [ex for ex in train_dataset if ex["answer"] == "NO"][:10]
tiny_dataset = tiny_serve + tiny_not
random.shuffle(tiny_dataset)

print(f"   Tiny dataset: {len(tiny_dataset)} samples")
print(f"   - YES (serve): {len(tiny_serve)}")
print(f"   - NO (not serve): {len(tiny_not)}")

IGNORE_INDEX = -100

def fixed_collate_fn(examples):
    """FIXED: Only compute loss on answer tokens, mask everything else."""
    texts = []
    images_list = []
    prompt_token_lens = []
    
    for example in examples:
        # Build prompt (without answer) to measure where answer starts
        prompt_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example["image"]},
                    {"type": "text", "text": example["question"]}
                ]
            }
        ]
        prompt_text = processor.apply_chat_template(
            prompt_messages, 
            tokenize=False, 
            add_generation_prompt=True  # Adds "assistant:" prefix
        )
        
        # Build full text (prompt + answer)
        full_messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": example["image"]},
                    {"type": "text", "text": example["question"]}
                ]
            },
            {
                "role": "assistant",
                "content": example["answer"]
            }
        ]
        full_text = processor.apply_chat_template(
            full_messages, 
            tokenize=False, 
            add_generation_prompt=False
        )
        
        texts.append(full_text)
        
        # Process image
        image_inputs, _ = process_vision_info(full_messages)
        images_list.append(image_inputs[0] if image_inputs else None)
        
        # We'll calculate token positions after processing
        prompt_token_lens.append(prompt_text)
    
    # Tokenize full batch
    batch = processor(
        text=texts,
        images=images_list,
        return_tensors="pt",
        padding=True,
        truncation=False,
    )
    
    # Create labels - ONLY compute loss on answer tokens
    labels = batch["input_ids"].clone()
    
    # For each example, mask everything except the answer
    for i, prompt_text in enumerate(prompt_token_lens):
        # Tokenize prompt separately to find where answer starts
        # Note: We process with image to get correct token positions
        prompt_batch = processor(
            text=[prompt_text],
            images=[images_list[i]],
            return_tensors="pt",
            padding=False,
            truncation=False,
        )
        prompt_len = prompt_batch["input_ids"].shape[1]
        
        # Mask all prompt tokens (only train on answer)
        labels[i, :prompt_len] = IGNORE_INDEX
    
    # Also mask padding
    labels[labels == processor.tokenizer.pad_token_id] = IGNORE_INDEX
    
    batch["labels"] = labels
    
    # Debug: print how many tokens we're training on
    trainable = (labels != IGNORE_INDEX).sum().item()
    total = labels.numel()
    print(f"   [Batch] Training on {trainable}/{total} tokens ({100*trainable/total:.1f}%)")
    
    return batch

# Aggressive training config for overfitting
debug_config = SFTConfig(
    output_dir="./debug_overfit",
    num_train_epochs=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    learning_rate=1e-3,  # Even higher LR since we're only training on ~1 token
    warmup_steps=0,
    lr_scheduler_type="constant",
    logging_steps=5,
    save_strategy="no",
    eval_strategy="no",
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field=None,
    dataset_kwargs={"skip_prepare_dataset": True},
    max_steps=100,
)

debug_trainer = SFTTrainer(
    model=model,
    args=debug_config,
    train_dataset=tiny_dataset,
    data_collator=fixed_collate_fn,
)

print("\n🚀 Starting overfit test (100 steps max)...")
print("   Watch the loss - should drop quickly now!\n")

debug_trainer.train()

print("\n" + "="*60)
print("📊 DIAGNOSIS:")
final_loss = debug_trainer.state.log_history[-1].get('loss', 999)
print(f"   Final loss: {final_loss:.4f}")

if final_loss < 1:
    print("   ✅ SUCCESS: Model CAN learn!")
    print("   → The fix worked. Now update B4 with this collate_fn")
elif final_loss < 3:
    print("   ⚠️ PARTIAL: Model is learning")
    print("   → May need more steps or higher LR")
else:
    print("   ❌ Still not working")
    print("   → May need to target more LoRA layers")
print("="*60)

In [ ]:
# @title B5. Save the Fine-tuned Model
from google.colab import files
import shutil

# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to {OUTPUT_DIR}/")

# Create ZIP for download
shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
print(f"📦 Created {OUTPUT_DIR}.zip")

# Download
files.download(f"{OUTPUT_DIR}.zip")
print("⬇️ Downloading...")

---
## Part C: Evaluation

In [ ]:
# @title C0. Load Trained Model from Google Drive (Run this before C1)
import os
import zipfile
from google.colab import drive
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
import torch

# ========== CONFIGURE ==========
MODEL_ZIP_PATH = "/content/drive/MyDrive/Model/serve-detector-lora-v5.zip"  # @param {type:"string"}
LOCAL_MODEL_DIR = "serve-detector-lora-v5"  # @param {type:"string"}
# ================================

# Mount Google Drive if not already mounted
if not os.path.exists("/content/drive"):
    print("📂 Mounting Google Drive...")
    drive.mount('/content/drive')

# Check zip exists
if not os.path.exists(MODEL_ZIP_PATH):
    print(f"❌ Model not found at: {MODEL_ZIP_PATH}")
    raise FileNotFoundError(f"Update MODEL_ZIP_PATH to your model location")

# Extract model
print(f"📦 Extracting model from: {MODEL_ZIP_PATH}")
with zipfile.ZipFile(MODEL_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall('.')

# Find the adapter_config.json to locate the actual model directory
import glob
adapter_configs = glob.glob("**/adapter_config.json", recursive=True)
if not adapter_configs:
    print("❌ No adapter_config.json found in extracted files!")
    print("Extracted contents:")
    !ls -la
    raise FileNotFoundError("adapter_config.json not found")

# Use the directory containing adapter_config.json
LOCAL_MODEL_DIR = os.path.dirname(adapter_configs[0]) or "."
print(f"✅ Found adapter at: {LOCAL_MODEL_DIR}/")

# Load base model with 4-bit quantization
print("\n🤖 Loading base model (Qwen2.5-VL-7B)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

# Load LoRA adapter
print(f"🔧 Loading LoRA adapter from: {LOCAL_MODEL_DIR}/")
model = PeftModel.from_pretrained(model, LOCAL_MODEL_DIR)

# Load processor
processor = AutoProcessor.from_pretrained(LOCAL_MODEL_DIR)

print("\n✅ Fine-tuned model loaded! Ready to run C1.")

In [ ]:
# @title C1. Test on Test Set (SAME PROMPT AS TRAINING)
import json
from PIL import Image
import re

# Load the SAME prompt used for training
try:
    with open("training_prompt.txt") as f:
        TEST_PROMPT = f.read().strip()
except:
    TEST_PROMPT = "Is this a pickleball serve situation? Answer YES or NO."

print(f"Using prompt: \"{TEST_PROMPT}\"\n")

with open("test_data.json") as f:
    test_data = json.load(f)

print(f"Evaluating on {len(test_data)} test samples...\n")

# Metrics
correct = 0
true_positives = 0
false_positives = 0
true_negatives = 0
false_negatives = 0

model.eval()

for item in test_data:
    image_path = item["image"]
    actual_label = item["label"]
    
    # Run inference with SAME prompt as training
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": TEST_PROMPT}  # SAME as training!
        ]
    }]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        # Use greedy decoding for consistency (no sampling)
        output_ids = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    
    response = processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    
    # Parse prediction - simple YES/NO
    response_clean = response.strip().upper()
    predicted_serve = response_clean.startswith("YES")
    actual_serve = actual_label == "serve"
    
    # Show each prediction
    status = "✅" if predicted_serve == actual_serve else "❌"
    print(f"{status} {os.path.basename(image_path):30} | Actual: {actual_label:10} | Pred: {response_clean[:20]}")
    
    # Update metrics
    if predicted_serve == actual_serve:
        correct += 1
    
    if predicted_serve and actual_serve:
        true_positives += 1
    elif predicted_serve and not actual_serve:
        false_positives += 1
    elif not predicted_serve and actual_serve:
        false_negatives += 1
    else:
        true_negatives += 1

# Calculate metrics
accuracy = correct / len(test_data)
precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print(f"\n{'='*50}")
print("📊 Evaluation Results")
print("="*50)
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"              SERVE    NOT_SERVE")
print(f"Actual SERVE   {true_positives:3d}        {false_negatives:3d}")
print(f"   NOT_SERVE   {false_positives:3d}        {true_negatives:3d}")
print(f"\nMetrics:")
print(f"   Accuracy:  {accuracy:.1%}")
print(f"   Precision: {precision:.1%}")
print(f"   Recall:    {recall:.1%}")
print(f"   F1 Score:  {f1:.1%}")
print("="*50)

In [ ]:
# @title C2. Test on Custom Image
from google.colab import files
from IPython.display import display
from PIL import Image

print("Upload an image to test:")
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename)
    img.thumbnail((400, 400))
    display(img)
    
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": filename},
            {"type": "text", "text": "Is this a pickleball serve situation? Answer YES or NO, then explain briefly."}
        ]
    }]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        return_tensors="pt",
    ).to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    
    response = processor.batch_decode(output_ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    
    print(f"\n🤖 Model response:")
    print(response)

---
## Part D: Using the Fine-tuned Model

After downloading `serve-detector-lora.zip`, you can use it in your project:

```python
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

# Load base model
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    device_map="auto",
)

# Load LoRA adapter
model = PeftModel.from_pretrained(base_model, "serve-detector-lora")
processor = AutoProcessor.from_pretrained("serve-detector-lora")

# Use for inference
# ... same as before
```